# Session 4: Data Cleaning
**Python for Data Analysis Workshop | World Development Indicators**

---

## Learning Objectives

By the end of this session you will be able to:
- Drop or fill missing values strategically
- Impute missing values with mean, median, or group statistics
- Fix data types (convert strings to numbers, numbers to categories)
- Standardise string columns (strip whitespace, fix case)
- Remove duplicate rows
- Create a clean, analysis-ready dataset

---

## 1. Load the Data

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("World_Dev_Indicators.csv")
print(f"Raw dataset: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")

# Always work on a copy — preserve the original
df = df_raw.copy()
df.head(3)

## 2. Handling Missing Values

Missing values are one of the most common data quality problems.
You have three main strategies:
1. **Drop** — remove rows or columns with missing data
2. **Fill with a constant** — replace NaN with 0, "Unknown", etc.
3. **Impute** — replace NaN with a calculated value (mean, median, or group stats)

### 2.1 Check What We Have to Clean

In [ ]:
# Reminder — missing value counts
print("Missing values before cleaning:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### 2.2 Strategy A: Drop Rows with Missing Values

Use this when rows with missing key data cannot be meaningfully analysed.

In [ ]:
# Drop rows missing GDP — creates a new DataFrame
df_gdp_clean = df.dropna(subset=["GDP (current US$)", "GDP per capita (current US$)"])
print(f"Rows after dropping GDP nulls: {len(df_gdp_clean):,} (dropped {len(df) - len(df_gdp_clean)} rows)")

In [ ]:
# dropna() with how and thresh arguments
# how="all" — only drops rows where ALL specified columns are null
# thresh=N  — keeps rows with at least N non-null values
df_thresh = df.dropna(thresh=9)   # keep rows with at least 9 non-null columns
print(f"Rows with thresh=9: {len(df_thresh):,}")

### 2.3 Strategy B: Fill with a Constant

In [ ]:
# Fill missing GDP with 0 — appropriate only if 0 means "no data reported"
df["GDP (current US$)"] = df["GDP (current US$)"].fillna(0)
print("NaNs in GDP after filling:", df["GDP (current US$)"].isnull().sum())

# Restore for next examples
df = df_raw.copy()

In [ ]:
# For categorical-style columns, fill with a placeholder string
df_filled = df.copy()
df_filled["Income Group"] = df_filled["Income Group"].fillna("Unknown")
print("Income Group value counts after fillna:")
print(df_filled["Income Group"].value_counts())

### 2.4 Strategy C: Impute with Mean or Median

Imputing with the column mean or median is better than 0 when the column represents a continuous metric.

In [ ]:
df = df_raw.copy()

# Impute Life Expectancy with the column median
le_col = "Life expectancy at birth, total (years)"
median_le = df[le_col].median()
print(f"Median life expectancy: {median_le:.1f} years")

df[le_col] = df[le_col].fillna(median_le)
print(f"NaNs after imputation: {df[le_col].isnull().sum()}")

### 2.5 Strategy D: Impute with Group Statistics (Best Practice)

Imputing with the **overall** mean ignores structure in the data.
A better approach: impute with the **group** mean (e.g., by Income Group or Region).

In [ ]:
df = df_raw.copy()
le_col = "Life expectancy at birth, total (years)"

# Compute group median by Income Group and Year
group_median = df.groupby(["Income Group","Year"])[le_col].transform("median")

# Fill only the NaN cells with the group median
df[le_col] = df[le_col].fillna(group_median)

# Any still-remaining NaNs? Fill with global median as a fallback
global_median = df_raw[le_col].median()
df[le_col] = df[le_col].fillna(global_median)

print(f"NaNs remaining in Life Expectancy: {df[le_col].isnull().sum()}")
print("\nSample after imputation:")
df[['Country','Year','Income Group',le_col]].head(8)

## 3. Fixing Data Types

In [ ]:
# The Year column is int — let us demonstrate converting it to a string
# (useful if you want to treat Year as a label, not a number for arithmetic)
df["Year_str"] = df["Year"].astype(str)
print("Year_str dtype:", df["Year_str"].dtype)
print("Year_str sample:", df["Year_str"].head(3).tolist())

# Convert back
df["Year_str"] = df["Year_str"].astype(int)

In [ ]:
# Create an ordered Categorical for Income Group
# This allows proper sorting by income level, not alphabetically
income_order = ["Low income", "Lower middle income", "Upper middle income", "High income", "Not classified"]
df["Income Group"] = pd.Categorical(
    df["Income Group"],
    categories=income_order,
    ordered=True
)
print("Income Group dtype:", df["Income Group"].dtype)
print("Sorted unique values:")
print(df["Income Group"].sort_values().unique())

## 4. Standardising String Columns

String (text) columns often have hidden problems:
- Leading or trailing whitespace
- Mixed upper/lower case
- Inconsistent abbreviations

In [ ]:
# The Region column has trailing whitespace — let us confirm and fix it
print("Before cleaning — unique regions (with repr):")
for r in df["Region"].unique():
    print(" ", repr(r))

# Fix: strip whitespace
df["Region"] = df["Region"].str.strip()

print("\nAfter cleaning:")
for r in sorted(df["Region"].unique()):
    print(" ", repr(r))

In [ ]:
# Standardise Country names: strip and convert to title case
# (Title Case means first letter of each word is capitalised)
df["Country"] = df["Country"].str.strip().str.title()
print("Sample country names after title case:")
print(df["Country"].head(10).tolist())

In [ ]:
# Create a clean Lending Type column — map codes to readable labels
lending_map = {
    "IDA": "IDA (soft loans)",
    "IBRD": "IBRD (market loans)",
    "Blend": "Blend",
    "Not classified": "Not classified"
}
df["Lending Type Clean"] = df["Lending Type"].map(lending_map)
print("Lending type mapping result:")
print(df[["Lending Type","Lending Type Clean"]].drop_duplicates())

## 5. Removing Duplicates

In [ ]:
print(f"Rows before deduplication: {len(df):,}")

# Remove fully duplicate rows
df = df.drop_duplicates()
print(f"Rows after dropping full duplicates: {len(df):,}")

# If you want to keep only the first occurrence of each Country+Year
# df = df.drop_duplicates(subset=["Country","Year"], keep="first")
# print(f"Rows after dropping key duplicates: {len(df):,}")

## 6. Putting It All Together — A Cleaning Pipeline

In practice, you chain all cleaning steps into a single, reproducible function.

In [ ]:
def clean_wdi(filepath):
    """Load and clean the World Development Indicators CSV."""
    df = pd.read_csv(filepath)

    # --- String standardisation ---
    df["Region"]  = df["Region"].str.strip()
    df["Country"] = df["Country"].str.strip().str.title()

    # --- Ordered categorical for Income Group ---
    income_order = ["Low income","Lower middle income","Upper middle income","High income","Not classified"]
    df["Income Group"] = pd.Categorical(df["Income Group"], categories=income_order, ordered=True)

    # --- Impute Life Expectancy by group median ---
    le_col = "Life expectancy at birth, total (years)"
    group_med = df.groupby(["Income Group","Year"])[le_col].transform("median")
    df[le_col] = df[le_col].fillna(group_med).fillna(df_raw[le_col].median())

    # --- Impute Population by group median ---
    pop_col = "Population, total"
    group_pop = df.groupby(["Income Group","Year"])[pop_col].transform("median")
    df[pop_col] = df[pop_col].fillna(group_pop)

    # --- Drop fully duplicate rows ---
    df = df.drop_duplicates()

    return df

df_clean = clean_wdi("World_Dev_Indicators.csv")
print(f"Clean dataset: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
print("\nMissing values in clean dataset:")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

In [ ]:
# Save the clean dataset for use in subsequent sessions
df_clean.to_csv("WDI_clean.csv", index=False)
print("Saved: WDI_clean.csv")

## 7. Summary Cheat Sheet

| Task | Code |
|------|------|
| Drop rows with NaN in column | `df.dropna(subset=['col'])` |
| Fill NaN with constant | `df['col'].fillna(0)` |
| Impute with global median | `df['col'].fillna(df['col'].median())` |
| Impute with group median | `df.groupby('grp')['col'].transform('median')` |
| Convert dtype | `df['col'].astype(int)` |
| Create ordered categorical | `pd.Categorical(s, categories=order, ordered=True)` |
| Strip whitespace | `df['col'].str.strip()` |
| Title case | `df['col'].str.title()` |
| Map values | `df['col'].map(dict)` |
| Remove duplicates | `df.drop_duplicates()` |

---

## 8. Exercises

1. Impute the missing `GDP (current US$)` values using the **regional** median (group by Region and Year). How many NaNs remain?
2. Create a new column `Income Level` that maps `Income Group` values to simplified labels: `'Low income'` → `'Low'`, middle groups → `'Middle'`, `'High income'` → `'High'`.
3. Standardise the `Lending Type` column so all values are in UPPER CASE.
4. **Bonus:** Write a function that generates a before/after comparison of null counts for any DataFrame.

In [ ]:
# Your answers here:
# Exercise 1

# Exercise 2

# Exercise 3

# Exercise 4 (bonus)
